### Retrain Crop-Based Classifiers

Fine-tunes the existing binary and/or 5-class classifiers with newly labeled crops.
Faster and needs fewer new samples than training from scratch.
Old weights are **backed up** to `outputs/training/model_runs/` before overwriting.

**Workflow:** label crops → move to `annotated_crops/` → **run this notebook**

**Input** — `data/training/annotated_crops/{class}/` · `models/binary_best.pth` · `models/5group_*.pth`  
**Output** — `outputs/training/model_runs/{RUN_NAME}_{ts}/` (logs + backup) · **overwrites** `models/binary_best.pth` and/or `models/5group_*.pth`

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `RETRAIN_BINARY` / `RETRAIN_5CLASS` | Set False to skip that model |
| `IMG_SIZE_BIN` | Shape mismatch crash if not 256 (must match `binary_best.pth` training size) |
| `IMG_SIZE_5CLS` | Shape mismatch crash if not 224 (must match 5-class model training size) |

**Optional (Cell 2):** `LR_FINETUNE` (1e-4 — keep lower than original 1e-3), `EPOCHS_BINARY` / `EPOCHS_5CLASS` (12), `FREEZE_BACKBONE` (True — set False for full fine-tune with large new dataset), `BATCH` (32), `DATASETS` ([] = all sub-folders)

**Background sampling:** balanced across camera plots — each plot contributes equally so busy plots cannot dominate. If crop filenames change, update only `parse_plot_key()` in the imports cell — see its docstring.


##### Cell 1 — Environment  *(no edits needed)*

**Local:** auto-detects the repo root via `git rev-parse --show-toplevel` — no path editing required.

**Colab:** uses the path extracted from the zip in Cell 0.

Sets all derived paths (`MODEL_DIR`, `LABELED_DIR`, output folders).

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print('✓ Already extracted')
    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    import subprocess as _sp
    _git_root  = Path(_sp.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
    BASE_DIR   = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'
    DRIVE_BASE = BASE_DIR

MODEL_DIR   = BASE_DIR / 'models'
LABELED_DIR = BASE_DIR / 'data' / 'training' / 'annotated_crops'
INSECTNET_W = BASE_DIR / 'InsectNet' / 'model.pth'
WEB_IMG_DIR = BASE_DIR / 'data' / 'web_images'

# Training outputs go to local SSD on Colab (fast); saved to Drive after training.
LOCAL_TRAINING = Path('/content/outputs/training') if IN_COLAB else BASE_DIR / 'outputs' / 'training'
LOCAL_TRAINING.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET_W: {INSECTNET_W}  exists={INSECTNET_W.exists()}')
print(f'WEB_IMG_DIR: {WEB_IMG_DIR}  exists={WEB_IMG_DIR.exists()}')


##### Cell 2 — Retrain config  ← **edit before retraining**

Key parameters:

- **`EPOCHS_BINARY`** / **`EPOCHS_5CLASS`** — training epochs. Fine-tuning typically
  needs fewer epochs than training from scratch. Start with 10–15.
- **`LR_FINETUNE`** — use a **smaller** value than initial training (1e-4 instead of
  1e-3) to avoid overwriting the pretrained features.
- **`BG_RATIO`** — background crops per insect crop in binary training.
- **`FREEZE_BACKBONE`** — if True, only the classifier head is updated. Use for small
  new datasets (< ~200 new crops). Set False for larger updates.
- **`RETRAIN_BINARY`** / **`RETRAIN_5CLASS`** — toggle which models to update.

In [ ]:
# ── Which sub-datasets to use ──────────────────────────────────────────
# Sub-folder names inside data/training/annotated_crops/.
# Leave empty [] to use ALL sub-folders automatically (excluding 'progress').
DATASETS = []   # e.g. ['labeled_ls', 'labeled_mb']

# ── Which models to retrain ──────────────────────────────────────────
RETRAIN_BINARY  = True
RETRAIN_5CLASS  = True

# ── Hyperparameters ───────────────────────────────────────────────
EPOCHS_BINARY   = 12     # epochs for binary classifier fine-tune
EPOCHS_5CLASS   = 12     # epochs for 5-class classifier fine-tune
LR_FINETUNE     = 1e-4   # lower LR for fine-tuning (was 1e-3 for initial training)
BATCH           = 32
IMG_SIZE_BIN    = 256    # must match binary_best.pth (EfficientNet-B2 binary)
IMG_SIZE_5CLS   = 224    # must match 5group_efficientnet.pth
BG_RATIO        = 3      # background:insect ratio for binary training
SEED            = 42

# ── Freeze backbone? ────────────────────────────────────────────────
# True  = only update the classifier head (faster, safer for small datasets)
# False = update the entire network (recommended if > ~500 new crops)
FREEZE_BACKBONE = False

# ── Class definitions ────────────────────────────────────────────────
CLASSES_BINARY  = ['background', 'insect']
CLASSES_5       = ['bumblebee', 'fly', 'butterfly', 'other', 'background']
INSECT_FOLDERS      = ['bumblebee', 'fly', 'butterfly', 'other']

# ── Web images for binary insect class ──────────────────────────────
WEB_DIR            = BASE_DIR / 'data' / 'web_images'  # iNaturalist images
USE_WEB_FOR_BINARY = True   # add web images as extra insect data for binary classifier

# Folder name -> canonical class (handles legacy naming)
ALIAS_5 = {
    'bumblebee':     'bumblebee',
    'fly':           'fly',
    'butterfly':     'butterfly',
    'butterfly':'butterfly',
    'other':         'other',
    'background':    'background',
}

print('Config ready.')
print(f'  Binary  : retrain={RETRAIN_BINARY}  epochs={EPOCHS_BINARY}  lr={LR_FINETUNE}  freeze={FREEZE_BACKBONE}')
print(f'  5-class : retrain={RETRAIN_5CLASS}  epochs={EPOCHS_5CLASS}  lr={LR_FINETUNE}  freeze={FREEZE_BACKBONE}')
# ── Resolve dataset sub-folders ─────────────────────────────────
if DATASETS:
    DATASET_DIRS = [LABELED_DIR / ds for ds in DATASETS]
else:
    DATASET_DIRS = sorted([d for d in LABELED_DIR.iterdir()
                           if d.is_dir() and d.name != 'progress'])
print(f'Datasets : {[d.name for d in DATASET_DIRS]}')


##### Cell 3 — Imports + training utilities

Loads PyTorch, torchvision, and all shared training functions. **Do not edit.**

In [ ]:
import shutil
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}  |  PyTorch: {torch.__version__}')
if torch.cuda.is_available(): print(f'GPU     : {torch.cuda.get_device_name(0)}')


def letterbox(img, size):
    w, h = img.size; ms = max(w, h)
    sq = Image.new('RGB', (ms, ms), (0, 0, 0))
    sq.paste(img, ((ms - w) // 2, (ms - h) // 2))
    return sq.resize((size, size), Image.BILINEAR)

class CropDataset(Dataset):
    def __init__(self, samples, tf): self.s = samples; self.tf = tf
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        p, l = self.s[i]; return self.tf(Image.open(p).convert('RGB')), l

def make_tf(sz, aug=False):
    if aug:
        return T.Compose([T.Lambda(lambda i: letterbox(i, sz)),
                          T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                          T.ColorJitter(0.3, 0.3, 0.2, 0.05),
                          T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    return T.Compose([T.Lambda(lambda i: letterbox(i, sz)),
                      T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def make_loader(samples, idxs, sz, batch, aug=False, weighted=True):
    sub = [samples[i] for i in idxs]; ds = CropDataset(sub, make_tf(sz, aug))
    if weighted and sub:
        labs = [s[1] for s in sub]; cnt = np.bincount(labs, minlength=max(labs)+1)
        wts = [1.0/max(1,cnt[l]) for l in labs]
        return DataLoader(ds, batch_size=batch,
                          sampler=WeightedRandomSampler(wts, len(wts)), num_workers=2, pin_memory=True)
    return DataLoader(ds, batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)

def stratified_split(samples, val_frac=0.2, test_frac=0.1, seed=42):
    rng = np.random.default_rng(seed); by_cls = defaultdict(list)
    for i, (_, l) in enumerate(samples): by_cls[l].append(i)
    tr, va, te = [], [], []
    for l, idxs in by_cls.items():
        rng.shuffle(idxs); n = len(idxs)
        nva = max(1, int(n*val_frac)); nte = max(1, int(n*test_frac))
        tr.extend(idxs[nva+nte:]); va.extend(idxs[nva:nva+nte]); te.extend(idxs[:nva])
    return tr, va, te

def collect_crops(dataset_dirs, classes, alias):
    """dataset_dirs: list of Path, each containing class subfolders."""
    ci = {c:i for i,c in enumerate(classes)}; smp = []
    for labeled_dir in dataset_dirs:
        for folder, cls in alias.items():
            if cls not in ci: continue
            dd = Path(labeled_dir)/folder
            if not dd.exists(): continue
            for ext in ('*.jpg','*.jpeg','*.png'):
                for p in dd.glob(ext): smp.append((p, ci[cls]))
    counts = {i:0 for i in range(len(classes))}
    for _,l in smp: counts[l] += 1
    return smp, counts

def weighted_criterion(counts, n, device):
    w = torch.tensor([1.0/max(1,counts.get(i,1)) for i in range(n)], dtype=torch.float, device=device)
    return nn.CrossEntropyLoss(weight=w/w.sum())

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, classes):
    model.eval(); ls=cor=tot=0; ap,al=[],[]
    tp={i:0 for i in range(len(classes))}; fp={i:0 for i in range(len(classes))}; fn={i:0 for i in range(len(classes))}
    for imgs,labels in loader:
        imgs,labels=imgs.to(device),labels.to(device); out=model(imgs); preds=out.argmax(1)
        ls+=criterion(out,labels).item()*labels.size(0)
        cor+=(preds==labels).sum().item(); tot+=labels.size(0)
        ap.extend(preds.cpu().tolist()); al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i]+=((preds==i)&(labels==i)).sum().item()
            fp[i]+=((preds==i)&(labels!=i)).sum().item()
            fn[i]+=((preds!=i)&(labels==i)).sum().item()
    pf={}
    for i in range(len(classes)):
        p2=tp[i]/max(1,tp[i]+fp[i]); r=tp[i]/max(1,tp[i]+fn[i])
        pf[classes[i]]=2*p2*r/max(1e-8,p2+r)
    return {'loss':ls/tot,'acc':cor/tot,'macro_f1':sum(pf.values())/max(1,len(classes)),'per_f1':pf,'preds':ap,'labels':al}

def load_efficientnet_for_finetune(ckpt_path, n_classes, freeze_backbone):
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        saved_classes = ckpt.get('classes', [])
        if len(saved_classes) == n_classes:
            model.load_state_dict(ckpt['state_dict'])
            print(f'  Loaded: {Path(ckpt_path).name}  '
                  f'classes={saved_classes}  val_f1={ckpt.get("val_macro_f1", "?")}')
        else:
            print(f'  WARNING: checkpoint has {len(saved_classes)} classes but model '
                  f'expects {n_classes}. Starting from ImageNet weights.')
    else:
        print(f'  No checkpoint at {ckpt_path}. Starting from ImageNet weights.')
    if freeze_backbone:
        for name, param in model.named_parameters():
            param.requires_grad = name.startswith('classifier')
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Backbone frozen — {trainable:,} trainable params (head only)')
    else:
        print(f'  Full network — {sum(p.numel() for p in model.parameters()):,} trainable params')
    return model

def run_finetune(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt, device, classes, model_dir, in_colab):
    crit=weighted_criterion(counts,len(classes),device)
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=lr)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    best_f1=0.0; hist={'tr':[],'va':[],'f1':[]}
    hdr=(f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"MacroF1":>8}  {"Acc":>6}  '
         +'  '.join(f'{c[:8]:>9}' for c in classes))
    print(f'\n{"="*70}\n{name}  epochs={epochs}  lr={lr}\n{"="*70}\n{hdr}')
    for ep in range(1,epochs+1):
        model.train(); tl=tc=tt=0
        for imgs,labels in tr_ldr:
            imgs,labels=imgs.to(device),labels.to(device)
            opt.zero_grad(); out=model(imgs); loss=crit(out,labels)
            loss.backward(); opt.step()
            tl+=loss.item()*labels.size(0); tc+=(out.argmax(1)==labels).sum().item(); tt+=labels.size(0)
        vr=eval_epoch(model,va_ldr,crit,device,classes); sched.step()
        new_best=vr['macro_f1']>best_f1
        if new_best:
            best_f1=vr['macro_f1']
            torch.save({'state_dict':model.state_dict(),'classes':classes,
                        'img_size':224,'val_macro_f1':best_f1,'epoch':ep},ckpt)
        pf=vr['per_f1']
        print(f'{ep:>4}  {tl/tt:>8.4f}  {vr["loss"]:>8.4f}  {vr["macro_f1"]:>8.3f}  {vr["acc"]:>6.3f}  '
              +'  '.join(f'{pf.get(c,0):>9.3f}' for c in classes)+('  *' if new_best else ''))
        hist['tr'].append(tl/tt); hist['va'].append(vr['loss']); hist['f1'].append(vr['macro_f1'])
    print(f'\nBest val macro-F1: {best_f1:.3f}  ->  {ckpt}')
    if in_colab:
        try: shutil.copy(str(ckpt), str(Path(model_dir)/Path(ckpt).name)); print('Drive backup ok')
        except Exception as e: print(f'Drive backup failed: {e}')
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
    ax1.plot(hist['tr'],label='train'); ax1.plot(hist['va'],label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'],lw=2,label='macro F1'); ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    curves_path=Path(model_dir)/f'{name}_retrain_curves.png'
    plt.savefig(curves_path,dpi=100); plt.close(); print(f'Curves: {curves_path}')
    return model

print('\u2713 Utilities loaded.')

##### Cell 4 — Inspect labeled data

Shows crop counts per class in `annotated_crops/`. The classifier needs at least ~50
crops per class to fine-tune reliably. If a class shows ⚠ low, add more crops via
`crop_labeler.py` before retraining.

In [ ]:
all_cls = ['background', 'bumblebee', 'fly', 'butterfly', 'other', 'unsure']
totals  = {cls: 0 for cls in all_cls}

print(f'{"Dataset":<15}  ' + '  '.join(f'{c[:6]:>6}' for c in all_cls))
print('-' * 75)
for ds_dir in DATASET_DIRS:
    row = []
    for cls in all_cls:
        d = ds_dir / cls
        n = sum(len(list(d.glob(ext))) for ext in ('*.jpg','*.jpeg','*.png')) if d.exists() else 0
        totals[cls] += n
        row.append(n)
    print(f'{ds_dir.name:<15}  ' + '  '.join(f'{n:>6}' for n in row))
print('-' * 75)
print(f'{"TOTAL":<15}  ' + '  '.join(f'{totals[c]:>6}' for c in all_cls))
print()
for cls in ['bumblebee','fly','butterfly','other']:
    t = totals[cls]
    status = '⚠ low (<30)' if t < 30 else ('ok' if t < 100 else 'good')
    print(f'  {cls:<15}: {t:>5}  {status}')


##### Cell 5 — Collect and split data

Loads all labeled crops from `annotated_crops/` and creates stratified train/val/test splits.

**Background sampling** is balanced across camera plots using `parse_plot_key()` + `sample_bg()`:
each plot contributes an equal quota so a busy plot cannot dominate the training set.
The bg quota is based on **field insect crop count only** (not web images).

> **If your crop filenames change** (new field season, different camera naming), update
> `parse_plot_key()` in the imports cell — that is the only function you need to touch.
> See its docstring for examples of other naming formats.


In [ ]:
# Binary data
insect_paths = []
bg_paths_all = []
for ds_dir in DATASET_DIRS:
    for folder in INSECT_FOLDERS:
        for ext in ('*.jpg','*.jpeg','*.png'): insect_paths.extend((ds_dir/folder).glob(ext))
    for ext in ('*.jpg','*.jpeg','*.png'): bg_paths_all.extend((ds_dir/'background').glob(ext))
_field_insect_n = len(insect_paths)  # field crops only — used for bg quota

# ── Add web images to binary insect class ────────────────────────
if USE_WEB_FOR_BINARY and WEB_DIR and Path(WEB_DIR).exists():
    _web_insect = []
    for _folder in INSECT_FOLDERS:
        for _ext in ('*.jpg','*.jpeg','*.png'):
            _web_insect.extend(Path(WEB_DIR).glob(f'{_folder}/{_ext}'))
    insect_paths.extend(_web_insect)
    print(f'Web images added to insect class: {len(_web_insect)}')
else:
    print('USE_WEB_FOR_BINARY=False or WEB_DIR not found — binary uses field crops only')

# bg quota based on field crops only — web images inflate insect count
# but bg crops come from field data, so ratio should match field distribution
bg_sampled  = sample_bg(bg_paths_all, _field_insect_n * BG_RATIO, SEED)
binary_data = [(p,1) for p in insect_paths] + [(p,0) for p in bg_sampled]
bin_counts  = {0:len(bg_sampled), 1:len(insect_paths)}
bin_tr, bin_va, bin_te = stratified_split(binary_data, seed=SEED)
print('Binary dataset:')
print(f'  insect     : {bin_counts[1]:>5}')
print(f'  background : {bin_counts[0]:>5}  (from {len(bg_paths_all)} available)')
print(f'  split      : train={len(bin_tr)}  val={len(bin_va)}  test={len(bin_te)}')

# 5-class data
smp_5, counts_5 = collect_crops(DATASET_DIRS, CLASSES_5, ALIAS_5)
tr_5, va_5, te_5 = stratified_split(smp_5, seed=SEED)
print('\n5-class dataset:')
for i, c in enumerate(CLASSES_5): print(f'  {c:20}: {counts_5.get(i,0):>5}')
print(f'  split      : train={len(tr_5)}  val={len(va_5)}  test={len(te_5)}')

##### Cell 6 — Back up existing models

Copies current weights to `*_prev.pth` before overwriting.

**To roll back:** `shutil.copy(MODEL_DIR/'binary_prev.pth', MODEL_DIR/'binary_best.pth')`

In [ ]:
import shutil
backups = [('binary_best.pth','binary_prev.pth'),('5group_efficientnet.pth','5group_efficientnet_prev.pth')]
for src_name, dst_name in backups:
    src = MODEL_DIR/src_name; dst = MODEL_DIR/dst_name
    if src.exists(): shutil.copy(str(src),str(dst)); print(f'  Backed up: {src_name} -> {dst_name}')
    else: print(f'  Skipped (not found): {src_name}')
print('\nBackup complete. Safe to retrain.')
def parse_plot_key(path):
    """
    Extract a per-camera-plot group key from a crop filename for stratified
    background sampling.  Returns a string that uniquely identifies one plot
    (camera location + date).  sample_bg() uses this to cap the contribution
    of any single busy plot so no location dominates the training set.

    ── CURRENT FILENAME FORMAT ──────────────────────────────────────────────
    Crops produced by infer_cropbased.ipynb are named:

      <site>__<camera>__<image>_crop<i>_<type>_<scope>.jpg

    where path components are joined with double-underscore (__) and the
    site token already encodes site, species, plot, and date, e.g.:

      Gruvan_Bal_p1_20250727__102_WSCT__WSCT1529_crop00_normal_roi.jpg
      └─ site token ──────┘  └─ cam ┘  └─ image+crop ───────────────┘

    Returns: everything before the first __ e.g. 'Gruvan_Bal_p1_20250727'

    ── LEGACY FORMAT (kept for backward compatibility) ──────────────────────
    Older crops may use the hdd_... convention:

      hdd_<n>_<year>_<site>_<species>_<plot>_[<date>_]<cam>__<image>_crop_<i>.jpg

    These filenames also contain __ so the current-format branch fires first,
    returning everything before the first __ (e.g. 'hdd_1_2025_cg_Vamy_p1_101_WSCT').
    Grouping is at camera level rather than plot level — still correct for sampling.

    ── HOW TO ADAPT FOR A NEW NAMING CONVENTION ─────────────────────────────
    Update only this function.  Keep the return value as a string that
    uniquely identifies one camera plot.  Everything else stays the same.

    Falls back to 'unknown' if the filename matches neither format —
    sampling still works but is no longer plot-balanced.
    """
    stem = Path(path).stem
    # Current format: site__camera__image... (double-underscore separators)
    if '__' in stem:
        return stem.split('__')[0]   # e.g. 'Gruvan_Bal_p1_20250727'
    # Legacy format without __ (rare / old exports)
    parts = stem.split('_')
    try:
        if parts[0] == 'hdd' and len(parts) > 6:
            return f'hdd{parts[1]}_{parts[3]}_{parts[4]}_{parts[5]}'
    except IndexError:
        pass
    return 'unknown'
def sample_bg(bg_paths, n_total, seed=42):
    """
    Sample n_total background crops balanced across camera plots.

    Uses parse_plot_key() to group crops by plot, then gives each plot an
    equal quota so that no single busy plot can dominate the training set.

    Args:
        bg_paths : list of Path — all available background crop paths
        n_total  : int — how many to sample in total
        seed     : int — RNG seed for reproducibility

    Prints a per-group breakdown so you can verify the balance.
    If a plot has fewer crops than its quota, it contributes all it has
    (the total sampled may be slightly below n_total in that case).
    """
    if not n_total:
        return []

    groups = {}
    for p in bg_paths:
        groups.setdefault(parse_plot_key(p), []).append(p)

    rng = np.random.default_rng(seed)
    for imgs in groups.values():
        rng.shuffle(imgs)

    n_groups  = len(groups)
    quota     = n_total // n_groups
    remainder = n_total  % n_groups

    print(f'Background sampling: {n_groups} plot group(s), quota={quota}/group')
    for key, imgs in sorted(groups.items()):
        print(f'  {key:30}: {len(imgs):>5} available')

    sampled = []
    for i, key in enumerate(sorted(groups)):
        take = quota + (1 if i < remainder else 0)
        sampled.extend(groups[key][:min(take, len(groups[key]))])

    rng.shuffle(sampled)
    print(f'Sampled {len(sampled)} background crops (target {n_total})')
    return sampled




##### Cell 7 — Retrain binary classifier

Fine-tunes `binary_best.pth`. Best checkpoint (by val macro-F1) is saved back.
Skip by setting `RETRAIN_BINARY = False` in Cell 2.

In [ ]:
if not RETRAIN_BINARY:
    print('RETRAIN_BINARY=False — skipping.')
else:
    print('Loading binary classifier for fine-tuning...')
    model_bin = load_efficientnet_for_finetune(
        MODEL_DIR/'binary_best.pth', n_classes=2, freeze_backbone=FREEZE_BACKBONE).to(DEVICE)
    tr_ldr_bin = make_loader(binary_data, bin_tr, IMG_SIZE_BIN, BATCH, aug=True)
    va_ldr_bin = make_loader(binary_data, bin_va, IMG_SIZE_BIN, BATCH)
    model_bin = run_finetune(model_bin,'binary_retrain',tr_ldr_bin,va_ldr_bin,
                             EPOCHS_BINARY,LR_FINETUNE,bin_counts,MODEL_DIR/'binary_best.pth',
                             DEVICE,CLASSES_BINARY,MODEL_DIR,IN_COLAB)
    te_ldr_bin = make_loader(binary_data, bin_te, IMG_SIZE_BIN, BATCH)
    te_bin = eval_epoch(model_bin,te_ldr_bin,weighted_criterion(bin_counts,2,DEVICE),DEVICE,CLASSES_BINARY)
    print(f'\nBinary test: MacroF1={te_bin["macro_f1"]:.3f}  Acc={te_bin["acc"]:.3f}')
    if HAS_SKLEARN: print(classification_report(te_bin['labels'],te_bin['preds'],target_names=CLASSES_BINARY,digits=3))

##### Cell 8 — Retrain 5-class EfficientNet

Fine-tunes `5group_efficientnet.pth`. Best checkpoint saved back.
Skip by setting `RETRAIN_5CLASS = False` in Cell 2.

In [ ]:
if not RETRAIN_5CLASS:
    print('RETRAIN_5CLASS=False — skipping.')
else:
    print('Loading 5-class EfficientNet for fine-tuning...')
    model_5cls = load_efficientnet_for_finetune(
        MODEL_DIR/'5group_efficientnet.pth', n_classes=5, freeze_backbone=FREEZE_BACKBONE).to(DEVICE)
    tr_ldr_5 = make_loader(smp_5, tr_5, IMG_SIZE_5CLS, BATCH, aug=True)
    va_ldr_5 = make_loader(smp_5, va_5, IMG_SIZE_5CLS, BATCH)
    model_5cls = run_finetune(model_5cls,'5class_retrain',tr_ldr_5,va_ldr_5,
                              EPOCHS_5CLASS,LR_FINETUNE,counts_5,MODEL_DIR/'5group_efficientnet.pth',
                              DEVICE,CLASSES_5,MODEL_DIR,IN_COLAB)
    te_ldr_5 = make_loader(smp_5, te_5, IMG_SIZE_5CLS, BATCH)
    te_5cls = eval_epoch(model_5cls,te_ldr_5,weighted_criterion(counts_5,5,DEVICE),DEVICE,CLASSES_5)
    print(f'\n5-class test: MacroF1={te_5cls["macro_f1"]:.3f}  Acc={te_5cls["acc"]:.3f}')
    if HAS_SKLEARN: print(classification_report(te_5cls['labels'],te_5cls['preds'],target_names=CLASSES_5,digits=3))

##### Cell 9 — Summary
Prints final model paths and next steps after retraining completes.

In [ ]:
print('='*55)
print('RETRAIN COMPLETE')
print('='*55)
updated = []
if RETRAIN_BINARY: updated.append('binary_best.pth')
if RETRAIN_5CLASS: updated.append('5group_efficientnet.pth')
if updated:
    print('\nUpdated models:')
    for name in updated:
        p = MODEL_DIR/name; size_mb = p.stat().st_size/1e6 if p.exists() else 0
        print(f'  ✓ {name:<40} ({size_mb:.1f} MB)')
    print('\nBackups saved as:')
    for name in updated:
        bk = MODEL_DIR/name.replace('.pth','_prev.pth')
        if bk.exists(): print(f'  ✓ {bk.name}')
print('\nNext steps:')
print('  1. Run infer_cropbased.ipynb with a new RUN_NAME to test the updated models')
print('  2. Run evaluate.ipynb to compare new vs previous results')
print('  3. If improved: delete *_prev.pth backups')
print('  4. If worse, roll back:')
print('       import shutil')
print('       shutil.copy(MODEL_DIR/"binary_prev.pth", MODEL_DIR/"binary_best.pth")')
print('       shutil.copy(MODEL_DIR/"5group_efficientnet_prev.pth", MODEL_DIR/"5group_efficientnet.pth")')